# Module 5: Trajectory Anomaly Detection (Isolation Forest, LOF, One-Class SVM)


In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append("../src")
from preprocessing import clean_gps_data
from feature_engineering import add_movement_features, daily_user_summary
from anomaly_detection import fit_isolation_forest, fit_local_outlier_factor, fit_one_class_svm, save_model

df_traj = pd.read_csv("../data/processed/indian_trajectories.csv")
df_clean = clean_gps_data(df_traj)
df_feat = add_movement_features(df_clean)
daily_df = daily_user_summary(df_feat)

anomaly_features = ["Total_Daily_Distance_KM", "Avg_Daily_Speed_KMH", "Max_Daily_Speed_KMH", "Night_Pings_Count", "Stop_Count"]

# Isolation Forest
iso_model, iso_labels, iso_scores = fit_isolation_forest(daily_df, anomaly_features)
daily_df["Isolation_Forest_Label"] = iso_labels
daily_df["Trajectory_Risk_Score"] = iso_scores

# LOF & One-Class SVM
lof_model, lof_labels, lof_scores = fit_local_outlier_factor(daily_df, anomaly_features)
daily_df["LOF_Label"] = lof_labels

print("=== Top 10 Anomalous Trajectory Days ===")
print(daily_df.sort_values(by="Trajectory_Risk_Score", ascending=False)[["User_ID", "Date_Str", "Total_Daily_Distance_KM", "Avg_Daily_Speed_KMH", "Night_Pings_Count", "Trajectory_Risk_Score"]].head(10))

save_model(iso_model, "../models/anomaly_model.pkl")
print("Saved models/anomaly_model.pkl")

